<a href="https://colab.research.google.com/github/KDY-RiskManager/risk-management-portfolio/blob/main/10-sql-portfolio-analysis/sql_portfolio_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# [Cell 1] Kaggle 데이터 다운로드 + SQLite DB 구축
# ============================================================
!pip install --quiet kaggle

import os
import sqlite3
import pandas as pd
import numpy as np
from getpass import getpass

os.environ['KAGGLE_USERNAME'] = input("Kaggle 사용자명(username)을 입력하세요: ")
os.environ['KAGGLE_KEY'] = getpass("Kaggle API 키를 입력하세요 (화면에 표시되지 않습니다): ")

!kaggle datasets download -d uciml/default-of-credit-card-clients-dataset
!unzip -o default-of-credit-card-clients-dataset.zip

df = pd.read_csv('UCI_Credit_Card.csv')
df = df.rename(columns={'default.payment.next.month': 'default'})

# ------------------------------------------------------------
# SQLite 데이터베이스 생성 + 테이블 적재
# ------------------------------------------------------------
# SQLite는 별도 서버 설치 없이 파이썬 표준 라이브러리로 바로 쓸 수 있는
# 파일 기반 SQL 엔진 -> Colab에서 SQL 실무 연습하기에 가장 간편한 선택
conn = sqlite3.connect('card_risk.db')
df.to_sql('card_accounts', conn, if_exists='replace', index=False)

print("=== card_accounts 테이블 생성 완료 ===")
check = pd.read_sql_query("SELECT COUNT(*) as 총건수 FROM card_accounts", conn)
print(check)

# ------------------------------------------------------------
# 테이블 구조 확인 (SQL로 직접 조회)
# ------------------------------------------------------------
schema = pd.read_sql_query("PRAGMA table_info(card_accounts)", conn)
print("\n=== 테이블 스키마 ===")
print(schema[['name', 'type']])

Kaggle 사용자명(username)을 입력하세요: KIMDAEYU
Kaggle API 키를 입력하세요 (화면에 표시되지 않습니다): ··········
Dataset URL: https://www.kaggle.com/datasets/uciml/default-of-credit-card-clients-dataset
License(s): CC0-1.0
100% 0.98M/0.98M [00:00<00:00, 120MB/s]

Archive:  default-of-credit-card-clients-dataset.zip
  inflating: UCI_Credit_Card.csv     
=== card_accounts 테이블 생성 완료 ===
     총건수
0  30000

=== 테이블 스키마 ===
         name     type
0          ID  INTEGER
1   LIMIT_BAL     REAL
2         SEX  INTEGER
3   EDUCATION  INTEGER
4    MARRIAGE  INTEGER
5         AGE  INTEGER
6       PAY_0  INTEGER
7       PAY_2  INTEGER
8       PAY_3  INTEGER
9       PAY_4  INTEGER
10      PAY_5  INTEGER
11      PAY_6  INTEGER
12  BILL_AMT1     REAL
13  BILL_AMT2     REAL
14  BILL_AMT3     REAL
15  BILL_AMT4     REAL
16  BILL_AMT5     REAL
17  BILL_AMT6     REAL
18   PAY_AMT1     REAL
19   PAY_AMT2     REAL
20   PAY_AMT3     REAL
21   PAY_AMT4     REAL
22   PAY_AMT5     REAL
23   PAY_AMT6     REAL
24    default  INTEGER


In [3]:
# ============================================================
# [Cell 2 - 수정] 포트폴리오 세그먼트별 현황 — SQL 집계 쿼리
# ============================================================
# default -> "default" 로 큰따옴표를 추가해 예약어 충돌 해결

query1 = """
SELECT
    CASE
        WHEN LIMIT_BAL < 50000 THEN '1. 5만 미만'
        WHEN LIMIT_BAL < 100000 THEN '2. 5~10만'
        WHEN LIMIT_BAL < 200000 THEN '3. 10~20만'
        WHEN LIMIT_BAL < 500000 THEN '4. 20~50만'
        ELSE '5. 50만 이상'
    END AS 한도구간,
    COUNT(*) AS 고객수,
    ROUND(SUM(LIMIT_BAL), 0) AS 총한도익스포저,
    ROUND(AVG("default") * 100, 2) AS 부실률_pct
FROM card_accounts
GROUP BY 한도구간
ORDER BY 한도구간
"""
result1 = pd.read_sql_query(query1, conn)
print("=== 한도구간별 포트폴리오 분포 ===")
print(result1)

query2 = """
SELECT
    CASE
        WHEN AGE < 30 THEN '20대'
        WHEN AGE < 40 THEN '30대'
        WHEN AGE < 50 THEN '40대'
        WHEN AGE < 60 THEN '50대'
        ELSE '60대 이상'
    END AS 연령대,
    COUNT(*) AS 고객수,
    ROUND(AVG(LIMIT_BAL), 0) AS 평균한도,
    ROUND(AVG("default") * 100, 2) AS 부실률_pct
FROM card_accounts
GROUP BY 연령대
ORDER BY 연령대
"""
result2 = pd.read_sql_query(query2, conn)
print("\n=== 연령대별 세그먼트 현황 ===")
print(result2)

query3 = """
SELECT
    CASE
        WHEN PAY_0 <= 0 THEN '0. 정상(연체없음)'
        WHEN PAY_0 = 1 THEN '1. 1개월 연체'
        WHEN PAY_0 = 2 THEN '2. 2개월 연체'
        ELSE '3. 3개월 이상 연체'
    END AS 연체구간,
    COUNT(*) AS 고객수,
    ROUND(SUM(LIMIT_BAL), 0) AS 총익스포저,
    ROUND(SUM(LIMIT_BAL) * 100.0 / (SELECT SUM(LIMIT_BAL) FROM card_accounts), 2) AS 익스포저비중_pct,
    ROUND(AVG("default") * 100, 2) AS 다음달부실전환율_pct
FROM card_accounts
GROUP BY 연체구간
ORDER BY 연체구간
"""
result3 = pd.read_sql_query(query3, conn)
print("\n=== 연체 심각도별 익스포저 집중도 ===")
print(result3)

=== 한도구간별 포트폴리오 분포 ===
        한도구간    고객수       총한도익스포저  부실률_pct
0   1. 5만 미만   4311  1.019820e+08    36.07
1   2. 5~10만   7139  4.528700e+08    26.01
2  3. 10~20만   7400  1.046820e+09    20.77
3  4. 20~50만  10222  2.938588e+09    15.49
4  5. 50만 이상    928  4.842700e+08    11.21

=== 연령대별 세그먼트 현황 ===
      연령대    고객수      평균한도  부실률_pct
0     20대   9618  124209.0    22.84
1     30대  11238  197001.0    20.25
2     40대   6464  180786.0    22.97
3     50대   2341  163909.0    24.86
4  60대 이상    339  187847.0    28.32

=== 연체 심각도별 익스포저 집중도 ===
           연체구간    고객수         총익스포저  익스포저비중_pct  다음달부실전환율_pct
0   0. 정상(연체없음)  23182  4.139924e+09       82.39         13.83
1     1. 1개월 연체   3688  5.487600e+08       10.92         33.95
2     2. 2개월 연체   2667  2.973560e+08        5.92         69.14
3  3. 3개월 이상 연체    463  3.849000e+07        0.77         71.92


한도구간별: 한도가 낮을수록(5만 미만 36.07%) 부실률이 높고, 한도가 높을수록(50만 이상 11.21%) 낮아지는 명확한 역상관 — 카드사가 신용도 낮은 고객에게 낮은 한도를 부여하는 게 실제로 리스크 관리 효과가 있다는 걸 보여주는 결과.

연령대별: 30대가 부실률 최저(20.25%)이고 20대와 60대 이상에서 높아지는 U자형 패턴 — 사회초년생(소득 불안정)과 고령층(은퇴 후 소득감소) 모두 리스크가 높다는 실무 직관과 일치.

연체구간별(가장 핵심 결과): 정상 고객의 다음달 부실전환율은 13.83%인데, 1개월 연체 고객은 33.95%(약 2.5배), 2개월 연체는 69.14%(5배), 3개월 이상은 71.92%로 뜀. 그런데 이 고위험군(1개월 이상 연체)이 전체 익스포저에서 차지하는 비중은 17.61%(10.92+5.92+0.77)뿐. **"소수의 연체 계좌가 압도적으로 높은 다음달 부실 위험을 안고 있다"**는 게 숫자로 명확히 드러났고, 이게 왜 카드사가 초기 연체(1~2개월)에서 집중 관리를 하는지에 대한 정량적 근거가 됨.

In [4]:
# ============================================================
# [Cell 3] Roll-rate 분석 — Window Function 활용
# ============================================================

# ------------------------------------------------------------
# 데이터를 "긴 형태(long format)"로 변환: 월별 연체상태를 세로로 나열
# ------------------------------------------------------------
# PAY_6(6개월전) -> PAY_5 -> ... -> PAY_0(최근월) 순서로 재구성
# SQL의 UNION ALL로 6개 컬럼을 하나의 시계열 컬럼으로 합침
query_long = """
SELECT ID, 1 AS month_seq, PAY_6 AS pay_status FROM card_accounts
UNION ALL
SELECT ID, 2 AS month_seq, PAY_5 AS pay_status FROM card_accounts
UNION ALL
SELECT ID, 3 AS month_seq, PAY_4 AS pay_status FROM card_accounts
UNION ALL
SELECT ID, 4 AS month_seq, PAY_3 AS pay_status FROM card_accounts
UNION ALL
SELECT ID, 5 AS month_seq, PAY_2 AS pay_status FROM card_accounts
UNION ALL
SELECT ID, 6 AS month_seq, PAY_0 AS pay_status FROM card_accounts
"""
df_long = pd.read_sql_query(query_long, conn)
df_long.to_sql('pay_history_long', conn, if_exists='replace', index=False)

# ------------------------------------------------------------
# LAG 윈도우 함수로 "직전월 연체상태"를 같은 행에 나란히 배치
# ------------------------------------------------------------
# LAG(컬럼, 1) OVER (PARTITION BY 고객 ORDER BY 시점): 고객별로 시간순 정렬 후
# 바로 이전 시점의 값을 가져옴 -> Roll-rate(상태전이) 분석의 핵심 기법
query_rollrate = """
WITH transitions AS (
    SELECT
        ID,
        month_seq,
        CASE WHEN pay_status <= 0 THEN '0.정상' ELSE '1.연체' END AS 현재상태,
        LAG(CASE WHEN pay_status <= 0 THEN '0.정상' ELSE '1.연체' END)
            OVER (PARTITION BY ID ORDER BY month_seq) AS 직전월상태
    FROM pay_history_long
)
SELECT
    직전월상태,
    현재상태,
    COUNT(*) AS 건수,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY 직전월상태), 2) AS 전이비율_pct
FROM transitions
WHERE 직전월상태 IS NOT NULL
GROUP BY 직전월상태, 현재상태
ORDER BY 직전월상태, 현재상태
"""
result_roll = pd.read_sql_query(query_rollrate, conn)
print("=== Roll-rate 매트릭스 (직전월 상태 -> 현재월 상태 전이율) ===")
print(result_roll)

# ------------------------------------------------------------
# 피벗: 보기 좋은 매트릭스 형태로 재구성
# ------------------------------------------------------------
roll_matrix = result_roll.pivot(index='직전월상태', columns='현재상태', values='전이비율_pct')
print("\n=== Roll-rate 매트릭스 (%) ===")
print(roll_matrix)

=== Roll-rate 매트릭스 (직전월 상태 -> 현재월 상태 전이율) ===
  직전월상태  현재상태      건수  전이비율_pct
0  0.정상  0.정상  123723     93.88
1  0.정상  1.연체    8069      6.12
2  1.연체  0.정상    4330     23.78
3  1.연체  1.연체   13878     76.22

=== Roll-rate 매트릭스 (%) ===
현재상태    0.정상   1.연체
직전월상태              
0.정상   93.88   6.12
1.연체   23.78  76.22


정상 고객은 93.88% 확률로 계속 정상을 유지하지만(신규 연체 발생률 6.12%), 일단 연체 상태가 된 고객은 76.22% 확률로 다음 달에도 연체가 지속됨(회복률은 23.78%뿐). 즉 연체는 한 번 시작되면 관성적으로 지속되는 경향이 3배 이상 강하다는 게 정량적으로 증명됨. 이게 카드사가 "연체 1개월차"에 조기 개입을 집중하는 이유를 데이터로 보여주는 핵심 결과.